In [101]:
from torchvision import models
from torchvision.models import vit_b_16

print("=" * 60)

try:
    m1 = models.resnet50(weights=None)
    print("ResNet50 build success")
except Exception as e:
    print("Fail", e)

try:
    m2 = vit_b_16(weights=None)
    print("Vit-B/16 build success")
except Exception as e:
    print("Fail",e)

print("=" * 60)

ResNet50 build success
Vit-B/16 build success


In [102]:
# 1. Import & Settings
import os
import gc
import random
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F

from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, f1_score, roc_auc_score, recall_score,
    roc_curve, ConfusionMatrixDisplay
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import vit_b_16, ViT_B_16_Weights, ResNet50_Weights

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

torch.backends.cudnn.benchmark = True

Device: cuda


In [103]:
print(f"현재 작업 경로: {os.getcwd()}")

test_relative = '/tf/nasw/dataset001/preprocessed/npz/fold_1_train.npz'
print(f"상대경로: {os.path.exists(test_relative)}")

현재 작업 경로: /tf/notebooks/Image Encoder/Untitled Folder
상대경로: False


In [104]:
#2. Hyperparameters & Paths
DATA_ROOT = Path("/tf/nasw/dataset001/preprocessed/npz_ct")
TABULAR_CSV_PATH = Path("/tf/nasw/dataset001/preprocessed/stage1_clinical_dataset.csv")

FOLDS = [1, 2, 3, 4, 5]

OUTPUT_DIR = Path("./ct_tabular_fusion_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_DIR = Path("./ct_binary_figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 8
NUM_WORKERS = 2
NUM_EPOCHS = 30
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 7

USE_SLICE_SUBSAMPLING = False

print("DATA_ROOT:", DATA_ROOT)
print("TABULAR_CSV_PATH:", TABULAR_CSV_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("FIGURE_DIR:", FIGURE_DIR.resolve())
print("FOLDS:", FOLDS)

DATA_ROOT: /tf/nasw/dataset001/preprocessed/npz_ct
TABULAR_CSV_PATH: /tf/nasw/dataset001/preprocessed/stage1_clinical_dataset.csv
OUTPUT_DIR: /tf/notebooks/Image Encoder/Untitled Folder/ct_tabular_fusion_models
FIGURE_DIR: /tf/notebooks/Image Encoder/Untitled Folder/ct_binary_figures
FOLDS: [1, 2, 3, 4, 5]


In [105]:
def inspect_fold_npz(fold_idx):
    train_npz_path = DATA_ROOT / f"fold_{fold_idx}_train.npz"
    val_npz_path = DATA_ROOT / f"fold_{fold_idx}_val.npz"

    print("TRAIN:", train_npz_path, train_npz_path.exits())
    print("VAL:", val_npz_path, val_npz_path.exists())

    train_npz = np.load(train_npz_path, allow_pickle=False, mmap_mode="r")
    val_npz = np.load(val_npz_path, allow_pickle=False, mmap_mode="r")

    for k in train_npz.files:
        print(f"[train] {k}: shape={train_npz[k].shape}, dtype={train_npz[k].dtype}")
    for k in val_npz.files:
        print(f"[val] {k}: shape={val_npz[k].shape}, dtype={val_npz[k].dtype}")

In [106]:
# local pretrained weight paths
LOCAL_RESNET50_IMAGENET_WEIGHTS = Path("/tf/pretrained_model/resnet50-11ad3fa6.pth")
LOCAL_VIT_WEIGHTS = Path("/tf/pretrained_model/vit_b_16-c867db91.pth")

print("LOCAL_RESNET50_IMAGENET_WEIGHTS exists:", LOCAL_RESNET50_IMAGENET_WEIGHTS.exists())
print("LOCAL_VIT_WEIGHTS exists:", LOCAL_VIT_WEIGHTS.exists())

LOCAL_RESNET50_IMAGENET_WEIGHTS exists: True
LOCAL_VIT_WEIGHTS exists: True


In [107]:
def load_local_state_dict(weight_path):
    weight_path = Path(weight_path)
    assert weight_path.exists(), f"Weight file not found: {weight_path}"

    ckpt = torch.load(weight_path, map_location="cpu")

    if isinstance(ckpt, dict) and "state_dict" in ckpt and isinstance(ckpt["state_dict"], dict):
        ckpt = ckpt["state_dict"]

    return ckpt

In [108]:
def compute_binary_metrics(y_true, y_prob, threshold = 0.5):
    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob).astype(float)
    y_pred = (y_prob > threshold).astype(int)

    metrics = {
        "acc": accuracy_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.5,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "cm": confusion_matrix(y_true, y_pred)
    }
    return metrics

In [109]:
def maybe_subsample_slices(x, is_train=True):
    if not USE_SLICE_SUBSAMPLING:
        return x

    S = x.shape[0]
    if S <= MAX_SLICES:
        return x

    if is_train:
        idx = np.sort(np.random.choice(S, size = MAX_SLICES, replace=False))
    else:
        idx = np.linspace(0, S-1, MAX_SLICES).round().astype(int)

    return x[idx]

def print_gpu_mem(prefix=""):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024 ** 3)
        reserv = torch.cuda.memory_reserved() / (1024 ** 3)
        print(f"{perfix} GPU alloc = {alloc:.2f} GB | reserved = {reserv:.2f} GB")

In [110]:
#TABULAR 파일 로드
TABULAR_STAGE1_DF = pd.read_csv(TABULAR_CSV_PATH)
TABULAR_STAGE1_DF["PT_ID"] = TABULAR_STAGE1_DF["PT_ID"].astype(str)

TABULAR_META_COLS = ["PT_ID", "fold", "split", "label", "recurrence"]
TABULAR_STAGE1_FEATURES = [c for c in TABULAR_STAGE1_DF.columns if c not in TABULAR_META_COLS]

# 1) bool -> int
bool_cols = TABULAR_STAGE1_DF[TABULAR_STAGE1_FEATURES].select_dtypes(include=["bool"]).columns.tolist()
for col in bool_cols:
    TABULAR_STAGE1_DF[col] = TABULAR_STAGE1_DF[col].astype(np.int8)

# 2) object 칼럼 처리
obj_cols = TABULAR_STAGE1_DF[TABULAR_STAGE1_FEATURES].select_dtypes(include=["object"]).columns.tolist()
for col in obj_cols:
    s = TABULAR_STAGE1_DF[col].astype(str).str.strip()

    lower_vals = set(s.dropna().str.lower().unique().tolist())

    if lower_vals.issubset({"True", "False"}):
        TABULAR_STAGE1_DF[col] = s.str.lower().map({"True":1, "False":0}).astype(np.int8)
    elif lower_vals.issubset({"0", "1"}):
        TABULAR_STAGE1_DF[col] = pd.to_numeric(s, error="raise").astype(np.int8)

    else:
        print(f"[WARN]아직 문자열 범주가 남아있는 컬럼: {col}")
        print("sample unique:", s.dropna().unique()[:10])


print("TABULAR_STAGE1_DF:", TABULAR_STAGE1_DF.shape)
print("TABULAR_STAGE1_FEATURES:", len(TABULAR_STAGE1_FEATURES))
print("tabular example cols:", TABULAR_STAGE1_FEATURES)

non_numeric_cols = TABULAR_STAGE1_DF[TABULAR_STAGE1_FEATURES].select_dtypes(exclude=[np.number]).columns.tolist()
print("tabular non_numeric_cols:", non_numeric_cols)

missing_total = TABULAR_STAGE1_DF[TABULAR_STAGE1_FEATURES].isna().sum().sum()
print("tabular missing values:", missing_total)

assert len(non_numeric_cols) == 0, f"csv에 numeric이 아닌 컬럼이 있습니다: {non_numeric_cols[:20]}"
assert missing_total == 0, "CSV에 결측치가 있습니다"

TABULAR_STAGE1_DF: (12810, 14)
TABULAR_STAGE1_FEATURES: 9
tabular example cols: ['Birth_YM', 'SONO_YM', 'PT_height', 'DIAG_ATT_AGE', 'CA_125', 'PT_GVD', 'PT_weight', 'CT_YM', 'PT_Para']
tabular non_numeric_cols: []
tabular missing values: 0


In [111]:
print(TABULAR_STAGE1_DF[["PT_ID", "fold", "split", "label"]].head())
print(TABULAR_STAGE1_DF["split"].value_counts(dropna=False))
print(TABULAR_STAGE1_DF["fold"].value_counts(dropna=False).sort_index())
print(TABULAR_STAGE1_DF["label"].value_counts(dropna=False).sort_index())

   PT_ID  fold  split  label
0  10008     1    val      1
1  10008     2  train      1
2  10008     3  train      1
3  10008     4  train      1
4  10008     5  train      1
split
train    10248
val       2562
Name: count, dtype: int64
fold
1    2562
2    2562
3    2562
4    2562
5    2562
Name: count, dtype: int64
label
0    10035
1     2775
Name: count, dtype: int64


In [112]:
class OvarianCTTabularNPZDataset(Dataset):
    def __init__(self, npz_path, tabular_df, is_train=False):
        super().__init__()
        self.is_train = is_train
        self.npz = np.load(npz_path, allow_pickle=False, mmap_mode="r")

        self.images = self.npz["images"]
        self.labels = self.npz["labels"].astype(np.float32)
        self.patient_ids = np.array([str(x) for x in self.npz["patient_ids"]])
        self.mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1)

        tab_df = tabular_df.copy()
        tab_df["PT_ID"] = tab_df["PT_ID"].astype(str)

        feature_cols = [c for c in tab_df.columns if c not in ["PT_ID", "fold", "split", "label", "recurrence"]] # recurrence도 제거
        self.feature_cols = feature_cols

        tab_map = {}
        for _, row in tab_df.iterrows():
            pid = str(row["PT_ID"])
            feat = row[feature_cols].values.astype(np.float32)
            lbl = float(row["label"])
            tab_map[pid] = (feat, lbl)

        valid_indices = []
        valid_tab_feats = []
        mismatch_count = 0

        for i, pid in enumerate(self.patient_ids):
            if pid not in tab_map:
                continue

            feat, tab_label = tab_map[pid]
            npz_label = float(self.labels[i])

            if abs(npz_label - tab_label) > 1e-6:
                mismatch_count += 1
                continue

            valid_indices.append(i)
            valid_tab_feats.append(feat)

        self.indices = np.array(valid_indices, dtype=np.int64)
        self.tabular_feats = (
            np.stack(valid_tab_feats, axis = 0)
            if len(valid_tab_feats) > 0
            else np.zeros((0, len(feature_cols)), dtype=np.float32)
        )

        print(f" total npz patients: {len(self.patient_ids)}")
        print(f" matched multimodal: {len(self.indices)}")
        print(f" label mismatch drop: {mismatch_count}")
        print(f" tabular feature dim: {len(self.feature_cols)}")
        

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        
        x = self.images[real_idx]
        y = self.labels[real_idx]
        tab = self.tabular_feats[idx]

        x = torch.from_numpy(x).float().permute(0, 3, 1, 2).contiguous()
        x = x / 255.0
        x = maybe_subsample_slices(x, is_train=self.is_train)
        x = (x-self.mean) / self.std

        tab = torch.tensor(tab, dtype=torch.float32)
        y = torch.tensor([y], dtype=torch.float32)

        return x, tab, y
    

In [113]:
# 7. fold별 dataLoader 생성
def make_fold_multimodal_loaders(fold_idx):
    train_npz_path = DATA_ROOT / f"fold_{fold_idx}_train.npz"
    val_npz_path = DATA_ROOT / f"fold_{fold_idx}_val.npz"

    assert train_npz_path.exists(), f"Missing: {train_npz_path}"
    assert val_npz_path.exists(), f"Missing: {val_npz_path}"

    train_tab_df = TABULAR_STAGE1_DF[
        (TABULAR_STAGE1_DF["fold"] == fold_idx) & (TABULAR_STAGE1_DF["split"] == "train")
    ].copy()

    val_tab_df = TABULAR_STAGE1_DF[
        (TABULAR_STAGE1_DF["fold"] == fold_idx) & (TABULAR_STAGE1_DF["split"] == "val")
    ].copy()

    train_dataset = OvarianCTTabularNPZDataset(train_npz_path, train_tab_df, is_train=True)
    val_dataset = OvarianCTTabularNPZDataset(val_npz_path, val_tab_df, is_train=False)

    train_loader = DataLoader(
        train_dataset,
        batch_size = BATCH_SIZE,
        shuffle = True,
        num_workers = NUM_WORKERS,
        pin_memory = (device.type == "cuda"),
        drop_last = True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size = BATCH_SIZE,
        shuffle = False,
        num_workers = NUM_WORKERS,
        pin_memory = (device.type == "cuda"),
        drop_last = False
    )

    labels_np = np.array([train_dataset.labels[i] for i in train_dataset.indices]).astype(int)
    neg = (labels_np == 0).sum()
    pos = (labels_np == 1).sum()

    pos_weight_value = neg / max(pos, 1)
    pos_weight = torch.tensor([pos_weight_value], device = device, dtype = torch.float32)

    tabular_in_dim = len(train_dataset.feature_cols)

    print(f"[FOLD {fold_idx}] Train benign(0): {neg}, malignant(1): {pos}, pos_weight={pos_weight.item():.4f}")
    print(f"[FOLD {fold_idx}] tabular_in_dim: {tabular_in_dim}")

    return train_loader, val_loader, pos_weight, tabular_in_dim

In [114]:
#9. 이미지 모델 정의
class PatientSliceAttentionClassifier(nn.Module):
    def __init__(self, encoder, feat_dim, hidden_dim=512, dropout=0.3):
        super().__init__()
        self.encoder = encoder

        self.attn = nn.Sequential(
            nn.Linear(feat_dim, feat_dim // 2),
            nn.Tanh(),
            nn.Linear(feat_dim // 2, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def encode(self, x):
        # x: (B, S, C, H, W)
        B, S, C, H, W = x.shape
        x = x.view(B * S, C, H, W)

        feat = self.encoder(x)
        feat = feat.view(B, S, -1)

        attn_score = self.attn(feat)
        attn_weight = torch.softmax(attn_score, dim=1)

        pooled = (feat * attn_weight).sum(dim=1)
        return pooled

    def forward(self, x):
        pooled = self.encode(x)
        logits = self.classifier(pooled)
        return logits

In [115]:
class CNNTransformerHybridPatient(nn.Module):
    def __init__(self,
                 d_model=512,
                 nhead=8, num_layers=2, 
                 dim_feedforward=1024, dropout=0.3, 
                 load_pretrained=True, freeze_backbone=True, unfreeze_layer4=True, 
                 resnet_weight_path=LOCAL_RESNET50_IMAGENET_WEIGHTS):
        super().__init__()
        backbone = models.resnet50(weights=None)
        if load_pretrained:
            state_dict = load_local_state_dict(resnet_weight_path)
            backbone.load_state_dict(state_dict, strict = True)

        feat_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.encoder = backbone
        
        if freeze_backbone:
            for p in self.encoder.parameters():
                p.requires_grad = False

            if unfreeze_layer4:
                for p in self.encoder.layer4.parameters():
                    p.requires_grad = True

        self.proj = nn.Linear(feat_dim, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1+8, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=0.1,
            batch_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers= num_layers)
        self.norm = nn.LayerNorm(d_model)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 1)
        )

    def encode(self, x):
        B, S, C, H, W = x.shape
        x = x.view(B * S, C, H, W)

        feat = self.encoder(x)
        feat = feat.view(B, S, -1)
        feat = self.proj(feat)

        cls = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, feat], dim=1)
        tokens = tokens + self.pos_embed[:, :tokens.size(1), :]

        tokens = self.transformer(tokens)
        tokens = self.norm(tokens)

        cls_out = tokens[:, 0, :]

        return cls_out

    def forward(self, x):
        cls_out = self.encode(x)
        logits = self.classifier(cls_out)
        return logits

In [116]:
 def build_resnet50_patient_model(load_pretrained=True, freeze_backbone=True, 
                                 unfreeze_layer4=True, 
                                 weight_path=LOCAL_RESNET50_IMAGENET_WEIGHTS):
    encoder = models.resnet50(weights=None)

    if load_pretrained:
        state_dict = load_local_state_dict(weight_path)
        encoder.load_state_dict(state_dict, strict = True)
        
    feat_dim = encoder.fc.in_features
    encoder.fc = nn.Identity()

    if freeze_backbone:
        for p in encoder.parameters():
            p.requires_grad = False

        if unfreeze_layer4:
            for p in encoder.layer4.parameters():
                p.requires_grad = True
    
    model = PatientSliceAttentionClassifier(
        encoder = encoder,
        feat_dim = feat_dim,
        hidden_dim = 512,
        dropout=0.1
    )
    
    return model

    
def build_vit_patient_model(load_pretrained=True, weight_path=LOCAL_VIT_WEIGHTS,
                            freeze_backbone=True, unfreeze_last_n_blocks=2):
    encoder = vit_b_16(weights=None)
    if load_pretrained:
        state_dict = load_local_state_dict(weight_path)
        encoder.load_state_dict(state_dict, strict = True)
    
    feat_dim = encoder.heads.head.in_features
    encoder.heads = nn.Identity()

    if freeze_backbone:
        for p in encoder.parameters():
            p.requires_grad = False
        for block in encoder.encoder.layers[-unfreeze_last_n_blocks:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in encoder.encoder.ln.parameters():
            p.requires_grad = True

    
    model = PatientSliceAttentionClassifier(
        encoder = encoder,
        feat_dim = feat_dim,
        hidden_dim = 512,
        dropout=0.1
    )
    return model
        

def build_hybrid_patient_model(load_pretrained=True):
    return CNNTransformerHybridPatient(
        load_pretrained=load_pretrained,
        freeze_backbone = True,
        unfreeze_layer4 = True,
        resnet_weight_path=LOCAL_RESNET50_IMAGENET_WEIGHTS
    )

In [117]:
#10) Tabular encoder + Fusion 모델 정의
class TabularMLPEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dims=(64,32), out_dim=64, dropout=0.2):
        super().__init__()

        layers = []
        prev_dim = in_dim

        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h),
                nn.ReLU(),
                nn.LayerNorm(h),
                nn.Dropout(dropout)
            ])
            prev_dim = h

        layers.append(nn.Linear(prev_dim, out_dim))
        self.encoder = nn.Sequential(*layers)

    def forward(self, x):
        return self.encoder(x)


class MultiModalBinaryFusionModel(nn.Module):
    def __init__(
        self,
        image_model,
        image_feat_dim,
        tabular_in_dim,
        tabular_feat_dim = 256,
        image_proj_dim=256,
        fusion_hidden_dim=256,
        dropout=0.3,
        tab_drop_p = 0.1,
        tab_scale = 0.3
    ):
        super().__init__()

        self.image_model = image_model
        self.tab_drop_p = tab_drop_p
        self.tab_scale = tab_scale

        self.tabular_encoder = TabularMLPEncoder(
            in_dim = tabular_in_dim,
            hidden_dims = (256, 128),
            out_dim = tabular_feat_dim,
            dropout=dropout
        )

        self.image_proj = nn.Sequential(
            nn.Linear(image_feat_dim, image_proj_dim),
            nn.ReLU(),
            nn.LayerNorm(image_proj_dim),
            nn.Dropout(dropout)
        )

        self.tab_proj = nn.Sequential(
            nn.Linear(tabular_feat_dim, tabular_feat_dim),
            nn.ReLU(),
            nn.LayerNorm(tabular_feat_dim),
            nn.Dropout(dropout)
        )

        self.fusion_head = nn.Sequential(
            nn.Linear(image_proj_dim + tabular_feat_dim, fusion_hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(fusion_hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden_dim, 1)
        )

    def forward(self, imgs, tabs):
        z_img = self.image_model.encode(imgs)
        z_img = self.image_proj(z_img)

        z_tab = self.tabular_encoder(tabs)
        z_tab = self.tab_proj(z_tab)

        if self.training and torch.rand(1).item() < self.tab_drop_p:
            z_tab = torch.zeros_like(z_tab)

        z_img = F.normalize(z_img, dim = 1)
        z_tab = F.normalize(z_tab, dim = 1)

        z_tab = self.tab_scale * z_tab

        z = torch.cat([z_img, z_tab], dim=1)
        logits = self.fusion_head(z)
        return logits

def build_resnet50_multimodal_model(tabular_in_dim, load_pretrained=True):
    image_model = build_resnet50_patient_model(load_pretrained = load_pretrained)

    model = MultiModalBinaryFusionModel(
        image_model = image_model,
        image_feat_dim = 2048,
        tabular_in_dim = tabular_in_dim,
        tabular_feat_dim=64,
        image_proj_dim=256,
        fusion_hidden_dim=256,
        dropout=0.2,
        tab_drop_p = 0.1,
        tab_scale = 0.3
    )

    return model

def build_vit_multimodal_model(tabular_in_dim, load_pretrained=True):
    image_model = build_vit_patient_model(load_pretrained = load_pretrained)

    model = MultiModalBinaryFusionModel(
        image_model = image_model,
        image_feat_dim = 768,
        tabular_in_dim = tabular_in_dim,
        tabular_feat_dim=64,
        image_proj_dim=256,
        fusion_hidden_dim=256,
        dropout=0.2,
        tab_drop_p = 0.1,
        tab_scale = 0.3
    )

    return model

def build_hybrid_multimodal_model(tabular_in_dim, load_pretrained=True):
    image_model = build_hybrid_patient_model(load_pretrained = load_pretrained)

    model = MultiModalBinaryFusionModel(
        image_model = image_model,
        image_feat_dim = 512,
        tabular_in_dim = tabular_in_dim,
        tabular_feat_dim=64,
        image_proj_dim=256,
        fusion_hidden_dim=256,
        dropout=0.2,
        tab_drop_p = 0.1,
        tab_scale = 0.3
    )

    return model

In [118]:
# 8. train/eval 함수

scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))

def train_one_epoch_multimodal(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    y_true, y_prob = [], []

    for imgs, tabs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        tabs = tabs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            logits = model(imgs, tabs)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)

        probs = torch.sigmoid(logits)
        y_true.extend(labels.detach().cpu().numpy().ravel())
        y_prob.extend(probs.detach().cpu().numpy().ravel())

    metric_vals = compute_binary_metrics(y_true, y_prob, threshold = 0.5)

    metrics = {
        "loss": running_loss / len(loader.dataset),
        "acc": metric_vals["acc"],
        "auc": metric_vals["auc"],
        "f1": metric_vals["f1"],
        "recall": metric_vals["recall"],
        "cm": metric_vals["cm"]
    }
    return metrics

@torch.no_grad()
def eval_one_epoch_multimodal(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    y_true, y_prob = [], []

    for imgs, tabs, labels in loader:
        imgs = imgs.to(device, non_blocking = True)
        tabs = tabs.to(device, non_blocking = True)
        labels = labels.to(device, non_blocking = True)

        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            logits = model(imgs, tabs)
            loss = criterion(logits, labels)

        running_loss += loss.item() * imgs.size(0)

        probs = torch.sigmoid(logits)
        y_true.extend(labels.detach().cpu().numpy().ravel())
        y_prob.extend(probs.detach().cpu().numpy().ravel())

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    metric_vals = compute_binary_metrics(y_true, y_prob, threshold = 0.5)

    metrics = {
        "loss": running_loss / len(loader.dataset),
        "auc": metric_vals["auc"],
        "acc": metric_vals["acc"],
        "f1": metric_vals["f1"],
        "recall": metric_vals["recall"],
        "cm": metric_vals["cm"]
    }
    return metrics

@torch.no_grad()
def eval_confusion_multimodal(model, loader, device, threshold = 0.5):
    model.eval()
    y_true, y_prob = [], []

    for imgs, tabs, labels in loader:
        imgs = imgs.to(device, non_blocking = True)
        tabs = tabs.to(device, non_blocking = True)
        labels = labels.to(device, non_blocking = True)

        logits = model(imgs, tabs)
        probs = torch.sigmoid(logits)

        y_true.extend(labels.cpu().numpy().ravel())
        y_prob.extend(probs.cpu().numpy().ravel())

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    metric_vals = compute_binary_metrics(y_true, y_prob, threshold = 0.5)

    print(f"\n--Final Evaluation (threshold={threshold}) --")
    print(f"Accuracy: {metric_vals['acc']:.4f}")
    print(f"ROC-AUC : {metric_vals['auc']:.4f}")
    print(f"F1-score : {metric_vals['f1']:.4f}")
    print(f"Sensitivity : {metric_vals['recall']:.4f}")
    print("Confusion Matrix:")
    print(metric_vals["cm"])

    final_metrics = {
        "auc": metric_vals["auc"],
        "acc": metric_vals["acc"],
        "f1": metric_vals["f1"],
        "recall": metric_vals["recall"],
        "cm": metric_vals["cm"],
    }

    print("\nClassification report:")
    y_pred = (np.array(y_prob) > threshold).astype(int)
    print(classification_report(np.array(y_true).astype(int), y_pred, digits = 4, zero_division = 0))

    return final_metrics, y_true, y_prob        

In [119]:
def find_best_threshold_for_sensitivity(y_true, y_prob, min_specificity=0.7):
    """
    Sensitivity가 최대가 되는 threshold 탐색
    min_specificity: 최소 specificity 조건
    """
    from sklearn.metrics import roc_curve

    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob).astype(float)

    fpr, tpr, thresholds = roc_curve(y_true, y_prob)

    best_threshold = 0.5
    best_sensitivity = 0.0

    for fpr_val, tpr_val, thr in zip(fpr, tpr, thresholds):
        specificity = 1.0 -fpr_val
        if specificity >= min_specificity and tpr_val > best_sensitivity:
            best_sensitivity = tpr_val
            best_threshold = thr

    return float(best_threshold), float(best_sensitivity)

In [120]:
def eval_with_optimal_threshold(model, loader, device, min_specificity=0.7):
    """
    val set에서 최적 threshold 탐색 후 전체 지표 반환
    """
    model.eval()
    y_true, y_prob = [], []

    with torch.no_grad():
        for imgs, tabs, labels in loader:
            imgs = imgs.to(device, non_blocking = True)
            tabs = tabs.to(device, non_blocking = True)
            labels = labels.to(device, non_blocking = True)

            logits = model(imgs,tabs)
            probs = torch.sigmoid(logits)

            y_true.extend(labels.cpu().numpy().ravel())
            y_prob.extend(probs.cpu().numpy().ravel())

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    best_thr, best_sens = find_best_threshold_for_sensitivity(
        y_true, y_prob, min_specificity=min_specificity
    )

    metrics_opt = compute_binary_metrics(y_true, y_prob, threshold=best_thr)
    metrics_default = compute_binary_metrics(y_true, y_prob, threshold=0.5)

    best_thr = float(best_thr)
    
    print(f"\n-- Threshold 비교--")
    print(f"{'':20s} {'threshold=0.5':>15s} {'optimal thr':>15s}")
    print(f"{'threshold':20s} {'0.5000':>15s} {f'th={best_thr:.4f}':>15s}")
    print(f"{'Sensitivity':20s} {metrics_default['recall']:>15.4f} {metrics_opt['recall']:>15.4f}")
    print(f"{'AUC':20s} {metrics_default['auc']:>15.4f} {metrics_opt['auc']:>15.4f}")
    print(f"{'F1':20s} {metrics_default['f1']:>15.4f} {metrics_opt['f1']:>15.4f}")
    print(f"{'ACC':20s} {metrics_default['acc']:>15.4f} {metrics_opt['acc']:>15.4f}")

    return {
        "threshold_default": 0.5,
        "threshold_optimal": best_thr,
        "sensitivity_default": metrics_default["recall"],
        "sensitivity_optimal": metrics_opt["recall"],
        "auc": metrics_default["auc"],
        "f1_default": metrics_default["f1"],
        "f1_optimal": metrics_opt["f1"],
        "acc_default": metrics_default["acc"],
        "acc_optimal": metrics_opt["acc"],
        "cm_default": metrics_default["cm"],
        "cm_optimal": metrics_opt["cm"],
    }

In [121]:
from sklearn.metrics import(
    roc_curve, auc, roc_auc_score,
    precision_recall_curve, average_precision_score
)

def _to_1d_prob(y_prob):
    y_prob = np.asarray(y_prob)

    if y_prob.ndim == 1:
        return y_prob
    elif y_prob.ndim == 2 and y_prob.shape[1] ==2:
        return y_prob[:, 1]
    else:
        raise ValueError(
            f"y_prob shape이 이상"
        )
def plot_cv_roc(
    fold_results,
    figsize=(6,6),
    roc_title="5-fold cv roc curve",
    save_roc_path=None,
    show_each_fold=True,
    ddof=1
):
    fold_aucs = []
    fold_aps = []

    all_y_true = []
    all_y_prob = []

    for i, fold in enumerate(fold_results, start=1):
        y_true=np.asarray(fold["y_true"]).astype(int)
        y_prob = _to_1d_prob(fold["y_prob"])

        if len(y_true) != len(y_prob):
            raise ValueError (f" {i}번째 fold: y_true y_prod 길이가 다름")
        if len(np.unique(y_true)) < 2:
            raise ValueError(f" {i}번째 fold: y_true에 클래스가 1개만")
        fold_auc = roc_auc_score(y_true, y_prob)
        fold_ap = average_precision_score(y_true, y_prob)

        fold_aucs.append(fold_auc)
        fold_aps.append(fold_ap)

        all_y_true.append(y_true)
        all_y_prob.append(y_prob)

    all_y_true = np.concatenate(all_y_true)
    all_y_prob = np.concatenate(all_y_prob)

    mean_auc = np.mean(fold_aucs)
    std_auc = np.std(fold_aucs)
    oof_auc = roc_auc_score(all_y_true, all_y_prob)

    plt.figure(figsize=figsize)
    
    if show_each_fold:
        for i, fold in enumerate(fold_results, start=1):
            y_true=np.asarray(fold["y_true"]).astype(int)
            y_prob = _to_1d_prob(fold["y_prob"])

            fpr, tpr, _ = roc_curve(y_true, y_prob)
            fold_auc = roc_auc_score(y_true, y_prob)

            plt.plot(
                fpr, tpr,
                alpha=0.35,
                label=f"Fold {i} (AUC={fold_auc:.4f})"
            )

    fpr_oof, tpr_oof, _ = roc_curve(all_y_true, all_y_prob)
    plt.plot(
        fpr_oof, tpr_oof,
        linewidth=2.5,
        label=f"OOF ROC (AUC={oof_auc:.4f})"
    )
    plt.plot([0,1], [0,1], linestyle="--", linewidth=1)

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{roc_title}\nMean AUC = {mean_auc:.4f} +- {std_auc:.4f}")
    plt.legend(loc="lower right", fontsize=9)
    plt.tight_layout()

    if save_roc_path is not None:
        plt.savefig(save_roc_path, dpi=150, bbox_inches="tight")

    plt.show()

    print("====5 fold cv roc summary === ")
    print("Fold AUCs:", [f"{x:.4f}" for x in fold_aucs])
    print(f"Mean AUC = {mean_auc:.4f} +- {std_auc:.4f}")
    print(f"OOF AUC = {oof_auc:.4f}")

    return {
        "fold_auc": fold_aucs,
        "mean_auc": mean_auc,
        "std_auc": std_auc,
        "oof_auc": oof_auc
    }


In [122]:
# 11) 공통 학습 함수
def fit_model_multimodal(model, model_name, fold_idx, train_loader, val_loader, device, 
              num_epochs=30, lr = 1e-4, weight_decay=1e-4, 
              pos_weight=None, patience = 7, min_delta=0.001):
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight = pos_weight)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr = lr,
        weight_decay = weight_decay
    )

    best_auc = -1
    best_val_loss = np.inf
    best_path = OUTPUT_DIR / f"{model_name}_fold{fold_idx}_best.pth"
    history = []
    early_stop_counter = 0

    for epoch in range(1, num_epochs+1):
        train_metrics = train_one_epoch_multimodal(model, train_loader, criterion, optimizer, device)
        val_metrics = eval_one_epoch_multimodal(model, val_loader, criterion, device)

        row = {
            "fold": fold_idx,
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["acc"],
            "train_f1": train_metrics["f1"],
            "train_recall": train_metrics["recall"],
            "train_auc": train_metrics["auc"],
            "val_loss":val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "val_f1": val_metrics["f1"],
            "val_auc": val_metrics["auc"],
            "val_recall": val_metrics["recall"],
        }
        history.append(row)

        print(
            f"[{model_name}][Fold {fold_idx}][Epoch {epoch:02d}]"
            f"train loss = {row['train_loss']:.4f}, acc={row['train_acc']:.4f}, auc={row['train_auc']:.4f},"
            f"f1 = {row['train_f1']:.4f}, recall={row['train_recall']:.4f} | "
            f"val loss = {row['val_loss']:.4f}, acc = {row['val_acc']:.4f}, auc = {row['val_auc']:.4f}, "
            f"f1 = {row['val_f1']:.4f}, recall={row['val_recall']:.4f}"
        )

        if row["val_auc"] > best_auc:
            best_auc = row["val_auc"]

        if row["val_loss"] < best_val_loss - min_delta:
            best_val_loss = row["val_loss"]
            early_stop_counter = 0
            torch.save(model.state_dict(), best_path)
            print(f"  -> best saved by val_loss: {best_path}")
        else:
            early_stop_counter += 1
            print(f"  -> no improvement ({early_stop_counter}/{patience})")

        if early_stop_counter >= patience:
            print(f"  -> early stopping triggered at epoch {epoch}")
            break

    history_df = pd.DataFrame(history)
    return model, history_df, best_path

In [123]:
# 12) model 로드 함수
def load_multimodal_model(model_type, model_path, device, tabular_in_dim):
    if model_type == "resnet":
        model = build_resnet50_multimodal_model(tabular_in_dim=tabular_in_dim, load_pretrained=True)
    elif model_type == "vit":
        model = build_vit_multimodal_model(tabular_in_dim=tabular_in_dim, load_pretrained=True)
    elif model_type == "hybrid":
        model = build_hybrid_multimodal_model(tabular_in_dim=tabular_in_dim, load_pretrained=True)
    else:
        raise ValueError("Unknown model type")
    

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    return model

In [124]:
# 13) 5-Fold CV
def run_cv_for_multimodal_model(model_name, model_type, build_fn):
    fold_results = []
    fold_histories = []
    roc_fold_results = []

    for fold_idx in FOLDS:
        print("\n" + "=" * 80)
        print(f"Running {model_name} | Fold {fold_idx}")
        print("=" * 80)

        train_loader, val_loader, pos_weight, tabular_in_dim = make_fold_multimodal_loaders(fold_idx)

        model = build_fn(tabular_in_dim)

        _, history_df, best_path = fit_model_multimodal(
            model=model,
            model_name=model_name,
            fold_idx = fold_idx,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            num_epochs=NUM_EPOCHS,
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            pos_weight=pos_weight,
            patience=PATIENCE
            
        )

        best_model = load_multimodal_model(model_type, best_path, device, tabular_in_dim)
        final_metrics, y_true_fold, y_prob_fold = eval_confusion_multimodal(best_model, val_loader, device, threshold=0.5)

        optimal_metrics = eval_with_optimal_threshold(
            best_model, val_loader, device, min_specificity = 0.8
        )

        print(f"\n[{model_name}][Fold {fold_idx}] Final Metrics")
        print(f"Accuracy: {final_metrics['acc']:.4f}")
        print(f"ROC-AUC: {final_metrics['auc']:.4f}")
        print(f"F1-score : {final_metrics['f1']:.4f}")
        print(f"Sensitivity : {final_metrics['recall']:.4f}")
        print("Confusion Matrix:")
        print(final_metrics["cm"])

        fold_results.append({
            "model": model_name,
            "fold": fold_idx,
            "acc": final_metrics["acc"],
            "ROC-AUC": final_metrics["auc"],
            "F1-score": final_metrics["f1"],
            "Sensitivity(0.5)": final_metrics["recall"],
            "Threshold_opt": optimal_metrics["threshold_optimal"],
            "Sensitivity_opt": optimal_metrics["sensitivity_optimal"],
            "f1_opt": optimal_metrics["f1_optimal"],
            "Confusion Matrix": final_metrics["cm"]
        })

        roc_fold_results.append({
            "y_true": y_true_fold,
            "y_prob": y_prob_fold
        })


        history_df["model"] = model_name
        history_df["fold"] = fold_idx
        fold_histories.append(history_df)

        del model
        del best_model
        del train_loader
        del val_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    result_df = pd.DataFrame(fold_results)
    history_df_all = pd.concat(fold_histories, ignore_index=True)

    roc_summary = plot_cv_roc(
        roc_fold_results,
        roc_title=f"{model_name} ROC Curve",
        save_roc_path=f"{model_name}_5fold_roc.png",
        show_each_fold=True,
        ddof = 1
    )
    
    #return result_df, history_df_all, roc_summary
    return result_df, history_df_all

In [125]:
#14) 실행
if __name__ == "__main__":
    
    print(">>> Training ResNet50...")
    resnet_cv_df, resnet_history_df = run_cv_for_multimodal_model(
        model_name="ResNet50_Fusion",
        model_type="resnet",
        build_fn=lambda tabular_in_dim: build_resnet50_multimodal_model(tabular_in_dim = tabular_in_dim, load_pretrained=True)
    )
    print(resnet_cv_df)
    
    print(">>> Training Vit...")
    vit_cv_df, vit_history_df = run_cv_for_multimodal_model(
        model_name="ViT_Fusion",
        model_type="vit",
        build_fn=lambda tabular_in_dim: build_vit_multimodal_model(tabular_in_dim = tabular_in_dim, load_pretrained=True)
    )
    print(vit_cv_df)
    
    print(">>> Training Hybrid...")
    hybrid_cv_df, hybrid_history_df = run_cv_for_multimodal_model(
        model_name="hybrid_Fusion",
        model_type="hybrid",
        build_fn=lambda tabular_in_dim: build_hybrid_multimodal_model(tabular_in_dim = tabular_in_dim, load_pretrained=True)
    )
    print(hybrid_cv_df)

    print("\\n>> 5-Fold Summary")
    all_cv_df = pd.concat([resnet_cv_df, vit_cv_df, hybrid_cv_df], ignore_index=True)

    summary_df = (
        all_cv_df.groupby("model")[["acc", "ROC-AUC", "F1-score", "Sensitivity"]].agg(["mean", "std"])
    )

    print(summary_df)
    
    
    

>>> Training ResNet50...

Running ResNet50_Fusion | Fold 1
 total npz patients: 2049
 matched multimodal: 2049
 label mismatch drop: 0
 tabular feature dim: 9
 total npz patients: 513
 matched multimodal: 513
 label mismatch drop: 0
 tabular feature dim: 9
[FOLD 1] Train benign(0): 1605, malignant(1): 444, pos_weight=3.6149
[FOLD 1] tabular_in_dim: 9
[ResNet50_Fusion][Fold 1][Epoch 01]train loss = 0.6433, acc=0.8042, auc=0.8942,f1 = 0.6378, recall=0.7950 | val loss = 0.3884, acc = 0.8616, auc = 0.9757, f1 = 0.7543, recall=0.9820
  -> best saved by val_loss: ct_tabular_fusion_models/ResNet50_Fusion_fold1_best.pth
[ResNet50_Fusion][Fold 1][Epoch 02]train loss = 0.3152, acc=0.9219, auc=0.9736,f1 = 0.8377, recall=0.9302 | val loss = 0.4460, acc = 0.9376, auc = 0.9817, f1 = 0.8462, recall=0.7928
  -> no improvement (1/7)
[ResNet50_Fusion][Fold 1][Epoch 03]train loss = 0.1773, acc=0.9629, auc=0.9913,f1 = 0.9183, recall=0.9617 | val loss = 0.4985, acc = 0.9513, auc = 0.9795, f1 = 0.8792, reca

KeyboardInterrupt: 

In [ ]:
def run_one_multimodal_model(model_name, model_type, build_fn):
    cv_df, history_df = run_cv_for_multimodal_model(
        model_name=model_name,
        model_type = model_type,
        build_fn = build_fn
    )
    display(cv_df)
    return cv_df, history_df

In [ ]:
hybrid_cv_df, hybrid_history_df = run_one_multimodal_model(
        model_name="hybrid_Fusion",
        model_type="hybrid",
        build_fn=lambda tabular_in_dim: build_hybrid_multimodal_model(tabular_in_dim = tabular_in_dim, load_pretrained=True)
    )

In [ ]:
def plot_one_model_by_fold(history_df, model_name):
    folds = sorted(history_df["fold"].unique())

    plt.figure(figsize=(14, 5))
    
    plt.subplot(1,3,1)
    for fold in folds:
        df_fold = history_df[history_df["fold"] == fold].sort_values("epoch")
        plt.plot(df_fold["epoch"], df_fold['train_loss'], linestyle='--', alpha = 0.7, label=f'Fold{fold} Train')
        plt.plot(df_fold["epoch"], df_fold['val_loss'], alpha = 0.7, label=f'Fold{fold} Val')
    plt.title(f"{model_name} - Loss by Fold")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend(fontsize=8)

    plt.subplot(1, 3, 2)
    for fold in folds:
        df_fold = history_df[history_df["fold"] == fold].sort_values("epoch")
        plt.plot(df_fold["epoch"], df_fold["val_auc"], alpha = 0.8, label = f"Fold{fold}")
    plt.title(f"{model_name} - Val ROC-AUC by Fold")
    plt.xlabel("Epoch")
    plt.ylabel("AUC")
    plt.legend(fontsize=8)

    plt.subplot(1, 3, 3)
    for fold in folds:
        df_fold = history_df[history_df["fold"] == fold].sort_values("epoch")
        plt.plot(df_fold["epoch"], df_fold["val_f1"], alpha = 0.8, label = f"Fold{fold}")
    plt.title(f"{model_name} - Val f1 by Fold")
    plt.xlabel("Epoch")
    plt.ylabel("f1")
    plt.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"{model_name}_training_curve.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
plot_one_model_by_fold(resnet_history_df, "ResNet50_Fusion")
plot_one_model_by_fold(vit_history_df, "ViT_Fusion")
plot_one_model_by_fold(hybrid_history_df, "hybrid_Fusion")

In [ ]:
@torch.no_grad()
def collect_predictions_multimodal(model, loader, device):
    model.eval()

    y_true, y_prob = [], []

    for imgs, tabs, labels in loader:
        imgs = imgs.to(device, non_blocking = True)
        tabs = tabs.to(device, non_blocking = True)
        labels = labels.to(device, non_blocking = True)

        logits = model(imgs, tabs)
        probs = torch.sigmoid(logits)

        y_true.extend(labels.cpu().numpy().ravel())
        y_prob.extend(probs.cpu().numpy().ravel())

    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob).astype(float)
    return y_true, y_prob

In [ ]:
from sklearn.metrics import roc_curve, auc

def plot_roc_curve(y_true, y_prob, title="ROC Curve", save_path = None):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6,6))
    plt.plot(fpr, tpr, label=f"AUC={roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")

    plt.show()

In [ ]:
def plot_pr_curve(y_true, y_prob, title = "Precision-Recall Curve", save_path = None):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)

    plt.figure(figsize=(6,6))
    plt.plot(recall, precision, label=f"AP = {ap:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title)
    plt.legend(loc="lower left")
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")

    plt.show()

In [ ]:
from sklearn.metrics import(
    roc_curve, auc, roc_auc_score,
    precision_recall_curve, average_precision_score
)

def _to_1d_prob(y_prob):
    y_prob = np.asarray(y_prob)

    if y_prob.ndim == 1:
        return y_prob
    elif y_prob.ndim == 2 and y_prob.shape[1] ==2:
        return y_prob[:, 1]
    else:
        raise ValueError(
            f"y_prob shape이 이상"
        )
def plot_cv_roc_pr(
    fold_results,
    figsize=(6,6),
    roc_title="5-fold cv roc curve",
    save_roc_path=None,
    show_each_fold=True,
    ddof=1
):
    fold_aucs = []
    fold_aps = []

    all_y_true = []
    all_y_prob = []

    for i, fold in enumerate(fold_results, start=1):
        y_true=np.asarray(fold["y_true"]).astype(int)
        y_prob = _to_1d_prob(fold["y_prob"])

        if len(y_true) != len(y_prob):
            raise ValueError (f" {i}번째 fold: y_true y_prod 길이가 다름")
        if len(np.unique(y_true)) < 2:
            raise ValueError(f" {i}번째 fold: y_true에 클래스가 1개만")
        fold_auc = roc_auc_score(y_true, y_prob)
        fold_ap = average_precision_score(y_true, y_prob)

        fold_aucs.append(fold_auc)
        fold_aps.append(fold_ap)

        all_y_true.append(y_true)
        all_y_prob.append(y_prob)

    all_y_true = np.concatenate(all_y_true)
    all_y_prob = np.concatenate(all_y_prob)

    mean_auc = np.mean(fold_aucs)
    std_auc = np.std(fold_aucs)
    oof_auc = roc_auc_score(all_y_true, all_y_prob)

    plt.figure(figsize=figsize)
    
    if show_each_fold:
        for i, fold in enumerate(fold_results, start=1):
            y_true=np.asarray(fold["y_true"]).astype(int)
            y_prob = _to_1d_prob(fold["y_prob"])

            fpr, tpr, _ = roc_curve(y_true, y_prob)
            fold_auc = roc_auc_score(y_true, y_prob)

            plt.plot(
                fpr, tpr,
                alpha=0.35,
                label=f"Fold {i} (AUC={fold_auc:.4f})"
            )

    fpr_oof, tpr_oof, _ = roc_curve(all_y_true, all_y_prob)
    plt.plot(
        fpr_oof, tpr_oof,
        linewidth=2.5,
        label=f"OOF ROC (AUC={oof_auc:.4f})"
    )
    plt.plot([0,1], [0,1], linestyle="--", linewidth=1)

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{roc_title}\nMean AUC = {mean_auc:.4f} +- {std_auc:.4f}")
    plt.legend(loc="lower right", fontsize=9)
    plt.tight_layout()

    if save_roc_path is not None:
        plt.savefig(save_roc_path, dpi=150, bbox_inches="tight")

    plt.show()

    print("====5 fold cv roc summary === ")
    print("Fold AUCs:", [f"{x:.4f}" for x in fold_aucs])
    print(f"Mean AUC = {mean_auc:.4f} +- {std_auc:.4f}")
    print(f"OOF AUC = {oof_auc:.4f}")

    return {
        "fold_auc": fold_aucs,
        "mean_auc": mean_auc,
        "std_auc": std_auc,
        "off_auc": off_auc
    }


In [ ]:
fold_results = [
    {"y_true": yt, "y_prob": yp}
    for yt, yp in zip(all_fold_y_true, all_fold_y_prob)
]

summary = plot_cv_roc(
    fold_results,
    roc_title="Hybrid Fusion ROC Curve",
    save_roc_path = "hybrid_5fold_roc.png",
    show_each_fold=True,
    ddof =1
)
    

In [ ]:
all_y_true = []
all_y_prob = []
fold_aucs = []
fold_aps = []

for fold in range(5):
    all_y_true.extend(y_val.tolist())
    all_y_prob.extend(y_prob.tolist())

    fold_aucs.append(roc_auc_score(y_val, y_prob))
    fold_aps.append(average_precision_score(y_val, y_prob))

print(f"Mean AUC: {np.mean(fold_aucs):.4f} +- {np.std(fold_aucs):.4f}")
print(f"Mean AP: {np.mean(fold_aps):.4f} +- {np.std(fold_aps):.4f}")

plot_roc_curve(
    np.array(all_y_true),
    np.array(all_y_prob),
    title=f"OOF ROC (mean AUC={np.mean(fold_aucs):.4f} +- {np.std(fold_aucs):.4f})"
)

plot_pr_curve(
    np.array(all_y_true),
    np.array(all_y_prob),
    title=f"OOF PR (mean AP={np.mean(fold_aps):.4f} +- {np.std(fold_aps):.4f})"
)


In [ ]:
@torch.no_grad()
def eval_ablation(model, loader, device, mode="full", threshold=0.5):
    model.eval()
    y_true, y_prob = [], []

    for imgs, tabs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        tabs = tabs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if mode == "img_only":
            tabs = torch.zeros_like(tabs)
        elif mode == "tab_only":
            imgs = torch.zeros_like(imgs)
        elif mode == "shuffled_tab":
            perm = torch.randperm(tabs.size(0), device=tabs.device)
            tabs = tabs[perm]

        logits = model(imgs, tabs)
        probs = torch.sigmoid(logits)

        y_true.extend(labels.cpu().numpy().ravel())
        y_prob.extend(probs.cpu().numpy().ravel())

    m = compute_binary_metrics(y_true, y_prob, threshold=threshold)
    print(mode, m["acc"], m["auc"], m["f1"], m["recall"])
    print(m["cm"])
    return m

best_path = OUTPUT_DIR / "ResNet50_Fusion_fold1_best.pth"
train_loader, val_loader, pos_weight, tabular_in_dim = make_fold_multimodal_loaders(1)

best_model = load_multimodal_model("resnet", best_path, device, tabular_in_dim)

eval_ablation(best_model, val_loader, device, mode="full")
eval_ablation(best_model, val_loader, device, mode="img_only")
eval_ablation(best_model, val_loader, device, mode="tab_only")
eval_ablation(best_model, val_loader, device, mode="shuffled_tab")

In [ ]:
fold_idx = 1
best_path = OUTPUT_DIR / "ResNet50_Fusion_fold1_best.pth"
train_loader, val_loader, pos_weight, tabular_in_dim = make_fold_multimodal_loaders(1)

best_model = load_multimodal_model("resnet", best_path, device, tabular_in_dim)

print("full val")
eval_loader(best_model, val_loader, device)

print("global")
shuffled_val_loader, perm = make_global_shuffled_val_loader(val_loader, seed=42)
eval_loader(best_model, shuffled_val_loader, device)

In [ ]:
class TabOverrideDataset(Dataset):
    def __init__(self, base_dataset, new_tabular_feats):
        self.base_dataset = base_dataset
        self.new_tabular_feats = np.asarray(new_tabular_feats, dtype=np.float32)

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        imgs, _, labels = self.base_dataset[idx]
        tabs = torch.tensor(self.new_tabular_feats[idx], dtype=torch.float32)
        return imgs, tabs, labels

def make_global_shuffled_val_loader(val_loader, seed=42):
    base_dataset = val_loader.dataset
    rng = np.random.default_rng(seed)

    perm = rng.permutation(len(base_dataset.tabular_feats))
    shuffled_tabular_feats = base_dataset.tabular_feats[perm].copy()

    shuffled_dataset = TabOverrideDataset(base_dataset, shuffled_tabular_feats)

    shuffled_loader = DataLoader(
        shuffled_dataset,
        batch_size = val_loader.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
        drop_last = False
    )

    return shuffled_loader, perm

@torch.no_grad()
def eval_loader(model, loader, device, threshold=0.5):
    model.eval()
    y_true, y_prob = [], []

    for imgs, tabs, labels in loader:
        imgs = imgs.to(device, non_blocking = True)
        tabs = tabs.to(device, non_blocking = True)
        labels = labels.to(device, non_blocking = True)

        logits = model(imgs, tabs)
        probs = torch.sigmoid(logits)

        y_true.extend(labels.cpu().numpy().ravel())
        y_prob.extend(probs.cpu().numpy().ravel())

    m = compute_binary_metrics(y_true, y_prob, threshold=threshold)
    print("acc:", m["acc"], "auc:", m["auc"], "f1:", m["f1"], m["recall"])
    print(m["cm"])
    return m

In [ ]:
fold_idx = 1
train_loader, val_loader, pos_weight, tabular_in_dim = make_fold_multimodal_loaders(fold_idx)

print("train dataset len:", len(train_loader.dataset))
print("val dataset len:", len(val_loader.dataset))

val_labels = [val_loader.dataset.labels[i] for i in val_loader.dataset.indices]
val_labels = np.array(val_labels).astype(int)

print("val benign:", (val_labels == 0).sum())
print("val malignant:", (val_labels == 1).sum())

In [ ]:
fold_idx = 1

train_tab = TABULAR_STAGE1_DF[
    (TABULAR_STAGE1_DF["fold"] == fold_idx) & (TABULAR_STAGE1_DF["split"] == "train")
].copy()
val_tab = TABULAR_STAGE1_DF[
    (TABULAR_STAGE1_DF["fold"] == fold_idx) & (TABULAR_STAGE1_DF["split"] == "val")
].copy()

train_ids = set(train_tab["PT_ID"].astype(str))
val_ids = set(val_tab["PT_ID"].astype(str))

print("TAB overlap:", len(train_ids & val_ids))
print(list(sorted(train_ids & val_ids))[:20])

In [ ]:
def plot_confusion_matrices_paper(model_info_list, save_path=None):
    n = len(model_info_list)
    ncols = 3
    nrow = (n + ncols -1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
    axes = axes.flatten()

    for i, info in enumerate(model_info_list):
        y_pred = (np.array(info["y_prob"]) > 0.5).astype(int)
        cm = confusion_matrix(info["y_true"], y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                      display_labels=["Benign", "Malignant"])
        disp.plot(ax=axes[i], colorbar=False, cmap="Blues")
        auc_val = roc_auc_score(info["y_true"], info["y_prob"])
        acc_val = accuracy_score(info["y_true"], y_pred)
        sens_val = recall_score(info["y_true"], y_pred, zero_division=0)
        axes[i].set_title(
            f'{infl["name"]}\nACC={acc_val:.3f}  AUC={auc_val:.3f}  Sens={sens_val:.3f}',
            fontsize=10, fontweight='bold'
        )

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
        
    fig.suptitle("Confusion Matrics - CT Binary Classification",
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"Confusion matrix figure saved -> {save_path}")
    plt.show()
    return fig